# 02 - Holt-Winters: Aditivo, Multiplicativo e Damped Trend

Este notebook explora o metodo de **Holt-Winters** usando a biblioteca **chronobox**.

**Conteudo:**
1. Holt-Winters aditivo vs multiplicativo
2. Damped trend (tendencia amortecida)
3. Comparacao no dataset airline.csv
4. Previsao com intervalos de confianca
5. Interpretacao dos parametros alpha, beta, gamma
6. Exercicios com brazil_ipca.csv

## 1. Fundamentacao Teorica

### Holt-Winters Aditivo

O metodo de Holt-Winters aditivo estende o metodo de Holt para incluir sazonalidade:

$$\hat{y}_{t+h|t} = l_t + h b_t + s_{t+h-m(k+1)}$$

Equacoes de atualizacao:
$$l_t = \alpha(y_t - s_{t-m}) + (1-\alpha)(l_{t-1} + b_{t-1})$$
$$b_t = \beta(l_t - l_{t-1}) + (1-\beta) b_{t-1}$$
$$s_t = \gamma(y_t - l_{t-1} - b_{t-1}) + (1-\gamma) s_{t-m}$$

### Holt-Winters Multiplicativo

Quando a amplitude sazonal cresce proporcionalmente ao nivel:

$$\hat{y}_{t+h|t} = (l_t + h b_t) \cdot s_{t+h-m(k+1)}$$

Equacoes de atualizacao:
$$l_t = \alpha \frac{y_t}{s_{t-m}} + (1-\alpha)(l_{t-1} + b_{t-1})$$
$$b_t = \beta(l_t - l_{t-1}) + (1-\beta) b_{t-1}$$
$$s_t = \gamma \frac{y_t}{l_{t-1} + b_{t-1}} + (1-\gamma) s_{t-m}$$

### Quando usar cada um?

| Caracteristica | Aditivo | Multiplicativo |
|----------------|---------|----------------|
| Amplitude sazonal | Constante | Proporcional ao nivel |
| Dados com zeros | Funciona | Problematico |
| Transformacao log | Nao precisa | Equivalente a aditivo no log |

## 2. Setup e Carregamento dos Dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys

from chronobox import HoltWinters, ETS

np.random.seed(42)

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')

# Carregar airline passengers
airline = pd.read_csv(os.path.join(DATA_DIR, 'airline.csv'), parse_dates=['date'])
airline.set_index('date', inplace=True)
y_airline = airline['passengers'].values

# Carregar IPCA
ipca = pd.read_csv(os.path.join(DATA_DIR, 'brazil_ipca.csv'), parse_dates=['date'])
ipca.set_index('date', inplace=True)
y_ipca = ipca['ipca'].values

print(f'Airline: {len(y_airline)} obs ({airline.index[0].strftime("%Y-%m")} a {airline.index[-1].strftime("%Y-%m")})')
print(f'IPCA: {len(y_ipca)} obs ({ipca.index[0].strftime("%Y-%m")} a {ipca.index[-1].strftime("%Y-%m")})')

## 3. Holt-Winters Aditivo

In [ ]:
# Ajustar Holt-Winters Aditivo no airline
hw_add = HoltWinters(seasonal='additive', seasonal_period=12)
res_add = hw_add.fit(y_airline)

print(res_add.summary())
print(f'\nParametros de suavizacao:')
print(f'  alpha (nivel): {res_add.alpha:.4f}')
print(f'  beta (tendencia): {res_add.beta:.4f}')
print(f'  gamma (sazonalidade): {res_add.gamma:.4f}')
print(f'\nMedricas de erro:')
print(f'  RMSE: {res_add.rmse:.2f}')
print(f'  MSE: {res_add.mse:.2f}')

In [ ]:
# Grafico 1: Ajuste HW Aditivo com componentes
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

axes[0].plot(y_airline, label='Observado', color='steelblue')
axes[0].plot(res_add.fittedvalues, label='HW Aditivo', color='darkorange', linestyle='--')
axes[0].set_title('Holt-Winters Aditivo - Airline Passengers')
axes[0].legend()
axes[0].set_ylabel('Passageiros')

axes[1].plot(res_add.level, color='green')
axes[1].set_title('Nivel ($l_t$)')
axes[1].set_ylabel('Nivel')

axes[2].plot(res_add.trend, color='purple')
axes[2].set_title('Tendencia ($b_t$)')
axes[2].set_ylabel('Tendencia')

axes[3].plot(res_add.season, color='crimson')
axes[3].set_title('Sazonalidade ($s_t$)')
axes[3].set_ylabel('Sazonal')
axes[3].set_xlabel('Observacao')

plt.tight_layout()
plt.show()

## 4. Holt-Winters Multiplicativo

In [ ]:
# Ajustar Holt-Winters Multiplicativo no airline
hw_mul = HoltWinters(seasonal='multiplicative', seasonal_period=12)
res_mul = hw_mul.fit(y_airline)

print(res_mul.summary())
print(f'\nParametros de suavizacao:')
print(f'  alpha (nivel): {res_mul.alpha:.4f}')
print(f'  beta (tendencia): {res_mul.beta:.4f}')
print(f'  gamma (sazonalidade): {res_mul.gamma:.4f}')
print(f'\nRMSE: {res_mul.rmse:.2f}')

## 5. Comparacao: Aditivo vs Multiplicativo

In [ ]:
# Grafico 2: Comparacao aditivo vs multiplicativo
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Ajustes sobrepostos
axes[0].plot(y_airline, label='Observado', color='steelblue', linewidth=1.5)
axes[0].plot(res_add.fittedvalues, label=f'HW Aditivo (RMSE={res_add.rmse:.2f})',
             color='darkorange', linestyle='--')
axes[0].plot(res_mul.fittedvalues, label=f'HW Multiplicativo (RMSE={res_mul.rmse:.2f})',
             color='green', linestyle='--')
axes[0].set_title('Comparacao: HW Aditivo vs Multiplicativo')
axes[0].legend()
axes[0].set_ylabel('Passageiros')

# Residuos
axes[1].plot(res_add.resid, label='Residuos Aditivo', color='darkorange', alpha=0.7)
axes[1].plot(res_mul.resid, label='Residuos Multiplicativo', color='green', alpha=0.7)
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('Comparacao de Residuos')
axes[1].legend()
axes[1].set_ylabel('Residuo')
axes[1].set_xlabel('Observacao')

plt.tight_layout()
plt.show()

print(f'RMSE Aditivo:        {res_add.rmse:.4f}')
print(f'RMSE Multiplicativo: {res_mul.rmse:.4f}')
melhor = 'Multiplicativo' if res_mul.rmse < res_add.rmse else 'Aditivo'
print(f'\nMelhor modelo: HW {melhor}')
print('\nA sazonalidade do airline cresce com o nivel, favorecendo o modelo multiplicativo.')

## 6. Damped Trend (Tendencia Amortecida)

A tendencia amortecida introduz um parametro $\phi \in (0,1)$ que **freia** a tendencia ao longo do tempo:

$$\hat{y}_{t+h|t} = l_t + (\phi + \phi^2 + \cdots + \phi^h) b_t + s_{t+h-m(k+1)}$$

Quando $h \to \infty$, a tendencia converge para $\frac{\phi}{1-\phi} b_t$.

**Por que usar damped trend?**
- Tendencias lineares sao irrealistas a longo prazo
- O damping evita previsoes explosivas
- Frequentemente melhora previsoes de longo prazo
- Gardner & McKenzie (1985) mostraram que damped trend supera tendencia linear em muitos casos

In [ ]:
# Comparacao: tendencia linear vs amortecida
# ETS com tendencia linear
ets_aa = ETS(error='A', trend='A', seasonal='A', seasonal_period=12, damped=False)
res_aa = ets_aa.fit(y_airline)

# ETS com tendencia amortecida
ets_ad = ETS(error='A', trend='A', seasonal='A', seasonal_period=12, damped=True)
res_ad = ets_ad.fit(y_airline)

print(f'{"Modelo":<25} {"AIC":>10} {"BIC":>10} {"phi":>8}')
print('-' * 55)
print(f'{"ETS(A,A,A)":<25} {res_aa.aic:>10.2f} {res_aa.bic:>10.2f} {"  -":>8}')
phi_val = f'{res_ad.phi:.4f}' if res_ad.phi is not None else '  -'
print(f'{"ETS(A,Ad,A) damped":<25} {res_ad.aic:>10.2f} {res_ad.bic:>10.2f} {phi_val:>8}')

In [ ]:
# Grafico 3: Impacto do damping nas previsoes
steps = 36
fc_linear = res_aa.forecast(steps=steps)
fc_damped = res_ad.forecast(steps=steps)

fig, ax = plt.subplots(figsize=(14, 6))

# Dados observados
ax.plot(range(len(y_airline)), y_airline, label='Observado', color='steelblue', linewidth=1.5)

# Previsao linear
idx_fc = range(len(y_airline), len(y_airline) + steps)
ax.plot(idx_fc, fc_linear, label='ETS(A,A,A) - Tendencia linear',
        color='darkorange', linewidth=2)
ax.plot(idx_fc, fc_damped, label='ETS(A,Ad,A) - Tendencia amortecida',
        color='green', linewidth=2, linestyle='--')

ax.axvline(len(y_airline) - 1, color='gray', linestyle=':', alpha=0.5)
ax.set_title('Impacto do Damping na Previsao (36 meses)')
ax.set_xlabel('Observacao')
ax.set_ylabel('Passageiros')
ax.legend()
plt.tight_layout()
plt.show()

print('A tendencia amortecida produz previsoes mais conservadoras.')
print('Para horizontes longos, o damping evita extrapolacoes excessivas.')

## 7. Previsao com Intervalos de Confianca

In [ ]:
# Previsao HW multiplicativo com intervalos
fc_mul = res_mul.forecast(steps=24)

# Previsao usando ETS equivalente para obter intervalos via simulacao
ets_mam = ETS(error='M', trend='A', seasonal='M', seasonal_period=12)
res_ets_mam = ets_mam.fit(y_airline)
fc_ets = res_ets_mam.forecast(steps=24)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(range(len(y_airline)), y_airline, label='Observado', color='steelblue', linewidth=1.5)
ax.plot(range(len(y_airline)), res_mul.fittedvalues, label='HW Multiplicativo ajustado',
        color='darkorange', linestyle='--', alpha=0.7)

idx_fc = range(len(y_airline), len(y_airline) + 24)
ax.plot(idx_fc, fc_mul, label='Previsao HW Multiplicativo',
        color='crimson', linewidth=2)

ax.axvline(len(y_airline) - 1, color='gray', linestyle=':', alpha=0.5)
ax.set_title('Holt-Winters Multiplicativo - Previsao 24 Meses')
ax.set_xlabel('Observacao')
ax.set_ylabel('Passageiros')
ax.legend()
plt.tight_layout()
plt.show()

print('Previsoes HW Multiplicativo (cada 6 meses):')
for i in range(0, 24, 6):
    print(f'  Passo {i+1:2d}: {fc_mul[i]:.1f}')

## 8. Interpretacao dos Parametros de Suavizacao

Os tres parametros controlam a **memoria** de cada componente:

### Alpha ($\alpha$) - Parametro de nivel
- Controla o peso das observacoes recentes no nivel
- $\alpha$ alto: nivel reage rapidamente a mudancas
- $\alpha$ baixo: nivel muda lentamente (mais suave)

### Beta ($\beta$) - Parametro de tendencia
- Controla a velocidade de mudanca da tendencia
- $\beta$ alto: tendencia muda rapido (captura quebras estruturais)
- $\beta$ baixo: tendencia estavel

### Gamma ($\gamma$) - Parametro de sazonalidade
- Controla a mutabilidade do padrao sazonal
- $\gamma$ alto: sazonalidade evolui ao longo do tempo
- $\gamma$ baixo: padrao sazonal quase fixo

In [ ]:
# Comparar parametros estimados entre aditivo e multiplicativo
print(f'{"Parametro":<12} {"HW Aditivo":>12} {"HW Multiplicativo":>18}')
print('-' * 45)
print(f'{"alpha":<12} {res_add.alpha:>12.4f} {res_mul.alpha:>18.4f}')
print(f'{"beta":<12} {res_add.beta:>12.4f} {res_mul.beta:>18.4f}')
print(f'{"gamma":<12} {res_add.gamma:>12.4f} {res_mul.gamma:>18.4f}')

print(f'\n{"Metrica":<12} {"HW Aditivo":>12} {"HW Multiplicativo":>18}')
print('-' * 45)
print(f'{"RMSE":<12} {res_add.rmse:>12.4f} {res_mul.rmse:>18.4f}')
print(f'{"MSE":<12} {res_add.mse:>12.4f} {res_mul.mse:>18.4f}')

# Indice sazonal
print('\nIndices sazonais (HW Multiplicativo):')
s = res_mul.season
meses = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
         'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
# Ultimos 12 indices sazonais
s_last = s[-12:]
for i, (m, v) in enumerate(zip(meses, s_last)):
    bar = '#' * int(abs(v) * 30) if isinstance(v, float) else ''
    print(f'  {m}: {v:>8.4f} {bar}')

## Exercicio

### Exercicio 1: Holt-Winters no IPCA brasileiro

Usando o dataset `brazil_ipca.csv`:

1. Plote a serie do IPCA e identifique tendencia e sazonalidade
2. Ajuste HW aditivo e multiplicativo com `seasonal_period=12`
3. Compare RMSE e analise os residuos
4. Qual metodo e mais adequado para inflacao? Justifique.

**Dica:** O IPCA pode ter valores negativos (deflacao), o que torna o modelo multiplicativo problematico. Verifique isso!

In [ ]:
# Exercicio 1 - Seu codigo aqui
# hw_ipca = HoltWinters(seasonal='additive', seasonal_period=12)
# res_ipca = hw_ipca.fit(y_ipca)
# ...

## Exercicio

### Exercicio 2: Damped Trend no IPCA

1. Ajuste ETS(A,A,A) e ETS(A,Ad,A) no IPCA com `seasonal_period=12`
2. Faca previsoes de 12 e 36 meses com cada modelo
3. Plote as previsoes lado a lado
4. Qual modelo e mais prudente para previsoes de inflacao? Por que?
5. Qual o valor estimado de $\phi$? O que ele indica?

In [ ]:
# Exercicio 2 - Seu codigo aqui
# ets_aaa = ETS(error='A', trend='A', seasonal='A', seasonal_period=12)
# ets_ada = ETS(error='A', trend='A', seasonal='A', seasonal_period=12, damped=True)
# ...